In [ ]:
!pip install 

In [1]:
# Cell 1 — Basic imports and setup
import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import ipywidgets as widgets
from IPython.display import display, clear_output

import tempfile, zipfile, json
from pathlib import Path
import geopandas as gpd
import time

In [2]:
# Initialize Earth Engine
ee.Authenticate()
ee.Initialize(
    project = "vegetation-monitoring-474607"
)

print("🔑 Earth Engine authenticated and initialized.")


🔑 Earth Engine authenticated and initialized.


In [5]:
# Create outputs directory
os.makedirs("./Outputs", exist_ok=True)

In [3]:
# Cell 2 — AOI input: upload (geojson/shapefile/kml) OR draw on map

# ---------- Helper: save uploaded file ----------
def save_uploaded_file(upload_widget):
    """Save uploaded file (works both in Colab and VS Code notebooks)."""
    if not upload_widget.value:
        print("⚠️ No file uploaded.")
        return None

    if isinstance(upload_widget.value, dict):  # classic
        uploaded = list(upload_widget.value.values())[0]
        name = uploaded["metadata"]["name"]
        content = uploaded["content"]
    else:  # VS Code or new Jupyter
        uploaded = upload_widget.value[0]
        name = uploaded.get("name") or uploaded.get("metadata", {}).get("name")
        content = uploaded.get("content")

    out_path = os.path.join(tempfile.gettempdir(), name)
    with open(out_path, "wb") as f:
        f.write(content)
    print("✅ Saved uploaded file to:", out_path)
    return out_path


# ---------- Helper: load AOI into Earth Engine ----------
def read_local_aoi(file_path):
    """Reads .zip/.shp/.geojson/.json/.kml → (ee.FeatureCollection, geopandas.GeoDataFrame)"""
    ext = os.path.splitext(file_path)[1].lower()

    # Load with geopandas
    if ext == ".zip":
        extract_dir = tempfile.mkdtemp()
        with zipfile.ZipFile(file_path, "r") as z:
            z.extractall(extract_dir)
        shp_files = [f for f in os.listdir(extract_dir) if f.endswith(".shp")]
        if not shp_files:
            raise ValueError("No .shp file found inside the ZIP archive.")
        gdf = gpd.read_file(os.path.join(extract_dir, shp_files[0]))

    elif ext in [".shp", ".geojson", ".json", ".kml"]:
        gdf = gpd.read_file(file_path)
    else:
        raise ValueError("Unsupported file type! Upload .zip, .shp, .geojson, or .kml")

    # Ensure CRS
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")
    else:
        gdf = gdf.to_crs(epsg=4326)

    # Convert to EE FeatureCollection safely
    ee_fc = geemap.geopandas_to_ee(gdf, geodesic=False)
    geom = ee_fc.geometry().buffer(0).simplify(30)
    print("✅ AOI loaded successfully from:", file_path)
    return ee.FeatureCollection(ee.Feature(geom)), gdf


# ---------- UI Widgets ----------
upload_widget = widgets.FileUpload(accept='.zip,.shp,.geojson,.json,.kml', multiple=False, description="Upload AOI")
upload_btn = widgets.Button(description="Load Uploaded AOI", button_style="success", icon="upload")
draw_btn = widgets.Button(description="Draw AOI on Map", button_style="info", icon="pencil")
out = widgets.Output(layout={'border': '1px solid #ccc', 'padding': '5px'})

display(widgets.HBox([upload_widget, upload_btn, draw_btn]))
display(out)

# ---------- Global holders ----------
aoi_fc = None
aoi_geom = None
gdf = None

# ---------- Callback: handle uploads ----------
def on_upload_clicked(b):
    with out:
        clear_output()
        try:
            local_path = save_uploaded_file(upload_widget)
            if not local_path:
                return
            fc, geo_df = read_local_aoi(local_path)
            globals()['aoi_fc'] = fc
            globals()['aoi_geom'] = fc.geometry()
            globals()['gdf'] = geo_df

            print("✅ AOI loaded into Earth Engine from upload.")
            print(f"Number of features: {len(geo_df)}")
            display(geo_df.head())

            # visualize
            m = geemap.Map()
            m.add_basemap("SATELLITE")
            m.addLayer(aoi_geom, {"color": "yellow"}, "AOI")
            m.center_object(aoi_geom)
            display(m)

        except Exception as e:
            print("❌ Error loading AOI:", e)

# ---------- Callback: draw AOI ----------
def on_draw_clicked(b):
    with out:
        clear_output()
        try:
            m = geemap.Map(center=[0, 0], zoom=2, height="500px")
            m.add_basemap("SATELLITE")
            m.add_draw_control()
            display(m)

            capture_btn = widgets.Button(description="Capture Drawn AOI", button_style="primary", icon="check")
            display(capture_btn)

            def capture(_):
                with out:
                    clear_output()
                    roi = m.user_roi
                    if roi is None:
                        print("⚠️ No AOI drawn. Please draw and click Capture again.")
                        display(m)
                        display(capture_btn)
                        return

                    # Ensure valid geometry: add a tiny buffer to fix zero-error-margin bug
                    try:
                        geom = roi.buffer(1, 1).simplify(30)
                    except Exception:
                        geom = roi.simplify(30)
                    
                    # Ensure it's a polygon
                    if geom.type().getInfo() != "Polygon":
                        geom = ee.Geometry.Polygon(geom.coordinates())
                    
                    fc = ee.FeatureCollection(ee.Feature(geom))

                    globals()['aoi_fc'] = fc
                    globals()['aoi_geom'] = geom

                    print("✅ AOI captured from map drawing.")
                    gdf_local = geemap.ee_to_gdf(fc)
                    globals()['gdf'] = gdf_local
                    display(gdf_local.head())

                    m.addLayer(geom, {"color": "yellow"}, "AOI (final)")
                    m.center_object(geom)
                    display(m)

            capture_btn.on_click(capture)

        except Exception as e:
            print("❌ Error drawing AOI:", e)

upload_btn.on_click(on_upload_clicked)
draw_btn.on_click(on_draw_clicked)

print("\n💡 Tip: You can either upload a shapefile/GeoJSON or draw directly on the map.")
print("Both methods will set the variable `aoi_geom` (ee.Geometry) and `aoi_fc` (ee.FeatureCollection).")


Output(layout=Layout(border_bottom='1px solid #ccc', border_left='1px solid #ccc', border_right='1px solid #cc…


💡 Tip: You can either upload a shapefile/GeoJSON or draw directly on the map.
Both methods will set the variable `aoi_geom` (ee.Geometry) and `aoi_fc` (ee.FeatureCollection).


In [4]:
# --- Parameters ---
start_date = "2024-01-01"
end_date = "2024-12-31"
scale = 10  # Sentinel-2 native resolution

print(f"📅 Using data from {start_date} to {end_date}")


📅 Using data from 2024-01-01 to 2024-12-31


In [5]:
# --- Cloud & Shadow Masking Function ---
def mask_s2_clouds(image):
    # Use QA60 band for cloud masking
    qa = image.select('QA60')
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
        qa.bitwiseAnd(cirrus_bit_mask).eq(0)
    )
    return image.updateMask(mask).divide(10000).copyProperties(image, ["system:time_start"])


In [6]:
# --- Compute NDVI & EVI ---
def add_indices(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    evi = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {
            'NIR': image.select('B8'),
            'RED': image.select('B4'),
            'BLUE': image.select('B2')
        }
    ).rename('EVI')
    return image.addBands([ndvi, evi])


In [7]:
# --- Load Sentinel-2 Collection (safe geometry handling) ---
aoi_geom = aoi_fc.geometry().buffer(0.001)  # add a small buffer to avoid zero-error issues

s2_sr = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(aoi_geom)
    .filterDate(start_date, end_date)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 50))
    .map(mask_s2_clouds)
    .map(add_indices)
)

print("🛰️ Images in filtered & masked collection:", s2_sr.size().getInfo())


🛰️ Images in filtered & masked collection: 95


In [8]:
# --- Optional: Compute VCI (Vegetation Condition Index) ---
def compute_vci(collection):
    ndvi = collection.select('NDVI')
    ndvi_min = ndvi.reduce(ee.Reducer.min())
    ndvi_max = ndvi.reduce(ee.Reducer.max())
    def add_vci(img):
        vci = img.expression(
            '((ndvi - ndvi_min) / (ndvi_max - ndvi_min)) * 100',
            {
                'ndvi': img.select('NDVI'),
                'ndvi_min': ndvi_min,
                'ndvi_max': ndvi_max
            }
        ).rename('VCI')
        return img.addBands(vci)
    return collection.map(add_vci)

s2_indices = compute_vci(s2_sr)

print("✅ Sentinel-2 with NDVI, EVI, and VCI ready.")

✅ Sentinel-2 with NDVI, EVI, and VCI ready.


In [9]:
# compute monthly AOI stats
def compute_monthly_stats_s2(collection, aoi, year=2024, scale=20):
    """
    Compute monthly mean, min, and max of NDVI, EVI, and VCI over the AOI.
    Optimized to avoid Earth Engine memory limits.
    """
    print(f"📅 Computing monthly NDVI/EVI/VCI stats for {year}...")
    results = []

    for m in range(1, 13):
        start = ee.Date.fromYMD(year, m, 1)
        end = start.advance(1, 'month')
        coll = collection.filterDate(start, end)

        count = coll.size().getInfo()
        if count == 0:
            results.append({'month': m, 'count': 0})
            print(f"→ Month {m:02d}: no images.")
            continue

        print(f"→ Month {m:02d}: {count} images... ", end="", flush=True)
        t0 = time.time()

        # Create mean composite
        mean_img = coll.mean().clip(aoi)

        # Compute AOI means for NDVI, EVI, VCI
        stats = mean_img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=aoi,
            scale=scale,
            maxPixels=1e10
        )

        stats_dict = stats.getInfo()
        row = {
            'month': m,
            'count': count,
            'mean_NDVI': stats_dict.get('NDVI'),
            'mean_EVI': stats_dict.get('EVI'),
            'mean_VCI': stats_dict.get('VCI'),
        }

        results.append(row)
        print(f"✅ done in {time.time()-t0:.1f}s")

    df = pd.DataFrame(results)
    print("\n✅ Monthly stats computation complete.")
    return df

# --- Run it ---
df_monthly = compute_monthly_stats_s2(s2_indices, aoi_geom, year=2024, scale=20)

# --- Save locally ---
os.makedirs("outputs", exist_ok=True)
csv_path = "outputs/sentinel2_monthly_veg_indices_2024.csv"
df_monthly.to_csv(csv_path, index=False)
print(f"💾 Saved results to {csv_path}")

display(df_monthly)


📅 Computing monthly NDVI/EVI/VCI stats for 2024...
→ Month 01: 5 images... ✅ done in 48.5s
→ Month 02: 11 images... ✅ done in 10.2s
→ Month 03: 11 images... ✅ done in 8.9s
→ Month 04: 6 images... ✅ done in 75.8s
→ Month 05: 5 images... ✅ done in 6.9s
→ Month 06: 8 images... ✅ done in 63.1s
→ Month 07: 5 images... ✅ done in 6.6s
→ Month 08: 6 images... ✅ done in 62.6s
→ Month 09: 10 images... ✅ done in 8.8s
→ Month 10: 10 images... ✅ done in 10.1s
→ Month 11: 8 images... ✅ done in 10.8s
→ Month 12: 10 images... ✅ done in 6.9s

✅ Monthly stats computation complete.
💾 Saved results to outputs/sentinel2_monthly_veg_indices_2024.csv


,month,count,mean_NDVI,mean_EVI,mean_VCI
0,1,5,0.551232,-0.487919,75.300960
1,2,11,0.536686,0.332119,72.503000
2,3,11,0.518926,0.124777,69.517443
3,4,6,0.558807,0.373786,76.308331
4,5,5,0.579205,-22.030303,79.975450
5,6,8,0.536764,0.573385,73.183878
6,7,5,0.455986,-53.696307,60.767525
7,8,6,0.472533,0.298392,62.780753
8,9,10,0.388022,0.240607,49.768163
9,10,10,0.414946,0.259010,53.875858


In [10]:
# --- Monthly Vegetation Trends (Individual Plots) ---
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import linregress
import os

# --- Prepare ---
df_plot = df_monthly.copy()
months = df_plot['month']
sns.set(style="whitegrid", palette="muted", font_scale=1.1)

# --- Define indices and colors ---
indices = {
    "NDVI": "green",
    "EVI": "blue",
    "VCI": "orange"
}

# --- Output folder ---
os.makedirs("outputs", exist_ok=True)

for idx_name, color in indices.items():
    col = f"mean_{idx_name}"

    fig, ax = plt.subplots(figsize=(10, 5))

    # --- Main line ---
    ax.plot(months, df_plot[col], marker='o', color=color, label=f"{idx_name} Mean")

    # --- Trendline ---
    y = df_plot[col].to_numpy()
    if np.all(np.isfinite(y)):
        slope, intercept, r_value, p_value, std_err = linregress(months, y)
        trend = intercept + slope * months
        ax.plot(months, trend, color=color, linestyle='--', alpha=0.7, label=f"{idx_name} Trend")

    # --- Labels & formatting ---
    ax.set_xticks(months)
    ax.set_xticklabels([f"{m:02d}" for m in months])
    ax.set_xlabel("Month")
    ax.set_ylabel("Index Value")
    ax.set_title(f"{idx_name} Monthly Trend ({start_date[:4]})")
    ax.legend()
    ax.grid(True)

    # --- Save figure ---
    fig_path = f"outputs/{idx_name.lower()}_trend_{start_date[:4]}.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"💾 Saved {idx_name} trend figure: {fig_path}")


💾 Saved NDVI trend figure: outputs/ndvi_trend_2024.png
💾 Saved EVI trend figure: outputs/evi_trend_2024.png
💾 Saved VCI trend figure: outputs/vci_trend_2024.png


#### Seasonal shading (e.g., highlighting wet vs. dry months)
#### Mean ± Standard Deviation bands (to show variability across months)

In [11]:
# Seasonal shading for Phenology Mapping
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import linregress
import os

# --- Prepare ---
df_plot = df_monthly.copy()
months = df_plot['month']
sns.set(style="whitegrid", palette="muted", font_scale=1.1)

# --- Define indices and colors ---
indices = {
    "NDVI": "green",
    "EVI": "blue",
    "VCI": "orange"
}

# --- Define wet/dry seasons (customize per region) ---
# Example for equatorial East Africa pattern:
#   - Wet: Mar–May and Oct–Dec
#   - Dry: Jan–Feb and Jun–Sep
wet_seasons = [(3, 5), (10, 12)]

# --- Output folder ---
os.makedirs("outputs", exist_ok=True)

for idx_name, color in indices.items():
    col_mean = f"mean_{idx_name}"
    col_std = f"std_{idx_name}" if f"std_{idx_name}" in df_plot.columns else None

    fig, ax = plt.subplots(figsize=(10, 5))

    # --- Seasonal shading ---
    for (start_m, end_m) in wet_seasons:
        ax.axvspan(start_m - 0.5, end_m + 0.5, color='skyblue', alpha=0.15)

    # --- Mean ± Std band ---
    y = df_plot[col_mean].to_numpy()
    if col_std:
        y_std = df_plot[col_std].to_numpy()
        ax.fill_between(months, y - y_std, y + y_std, color=color, alpha=0.2, label="±1σ")

    # --- Mean line ---
    ax.plot(months, y, marker='o', color=color, label=f"{idx_name} Mean")

    # --- Trendline ---
    if np.all(np.isfinite(y)):
        slope, intercept, r_value, p_value, std_err = linregress(months, y)
        trend = intercept + slope * months
        ax.plot(months, trend, color=color, linestyle='--', alpha=0.7, label=f"{idx_name} Trend")

    # --- Formatting ---
    ax.set_xticks(months)
    ax.set_xticklabels([f"{m:02d}" for m in months])
    ax.set_xlabel("Month")
    ax.set_ylabel("Index Value")
    ax.set_title(f"{idx_name} Monthly Trend ({start_date[:4]})")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # --- Annotate ---
    ax.text(0.02, 0.95, "Wet season shaded", transform=ax.transAxes, fontsize=9, color='gray', alpha=0.7)

    # --- Save figure ---
    fig_path = f"outputs/{idx_name.lower()}_trend_{start_date[:4]}.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"💾 Saved {idx_name} enhanced trend figure: {fig_path}")


💾 Saved NDVI enhanced trend figure: outputs/ndvi_trend_2024.png
💾 Saved EVI enhanced trend figure: outputs/evi_trend_2024.png
💾 Saved VCI enhanced trend figure: outputs/vci_trend_2024.png


In [12]:
def save_monthly_composites_with_colorbar(collection, aoi, index, year, scale=10):
    """
    Create monthly composites (mean, min, max) for a given vegetation index (NDVI/EVI/VCI)
    plus RGB and FalseColor composites.
    - Each month saved as a colorful PNG.
    - Summary 12-panel image per stat with a horizontal colorbar at the bottom.
    """

    import matplotlib.pyplot as plt
    import matplotlib as mpl
    import numpy as np
    import os
    import geemap
    import ee
    import tempfile
    import rasterio
    from matplotlib.colors import ListedColormap

    stats_types = ['mean', 'min', 'max']
    vis_params = {
        'NDVI': {'min': 0, 'max': 1, 'palette': ['red', 'yellow', 'green']},
        'EVI': {'min': -0.2, 'max': 1, 'palette': ['blue', 'yellow', 'green']},
        'VCI': {'min': 0, 'max': 100, 'palette': ['red', 'yellow', 'green']},
    }

    rgb_vis = {
        "true": {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
        "false": {'bands': ['B8', 'B4', 'B3'], 'min': 0, 'max': 3000}
    }

    out_dir = os.path.join("outputs", f"{index}_{year}")
    os.makedirs(out_dir, exist_ok=True)
    print(f"💾 Saving all outputs to: {out_dir}")

    # === MAIN COMPOSITES (NDVI/EVI/VCI) ===
    for stat in stats_types:
        print(f"\n📊 Processing {stat.upper()} composites for {index}...")

        fig, axes = plt.subplots(3, 4, figsize=(20, 12))
        axes = axes.flatten()

        for month in range(1, 13):
            start = ee.Date.fromYMD(year, month, 1)
            end = start.advance(1, 'month')
            coll = collection.filterDate(start, end)

            try:
                if coll.size().getInfo() == 0:
                    axes[month-1].text(0.5, 0.5, "No data", ha='center', va='center')
                    axes[month-1].set_title(f"{month:02d}")
                    axes[month-1].axis('off')
                    continue

                # --- Select composite type ---
                if stat == 'mean':
                    img = coll.mean().select(index).clip(aoi)
                elif stat == 'min':
                    img = coll.min().select(index).clip(aoi)
                else:
                    img = coll.max().select(index).clip(aoi)

                # --- Export image ---
                tmp_tif = os.path.join(tempfile.gettempdir(), f"{index}_{stat}_{month:02d}.tif")
                geemap.ee_export_image(
                    img,
                    filename=tmp_tif,
                    scale=scale,
                    region=aoi,
                    file_per_band=False
                )

                # --- Read and colorize image ---
                with rasterio.open(tmp_tif) as src:
                    band = src.read(1).astype(float)
                    band[band == 0] = np.nan

                vmin, vmax = vis_params[index]['min'], vis_params[index]['max']
                band_scaled = np.clip((band - vmin) / (vmax - vmin), 0, 1)
                cmap = ListedColormap(vis_params[index]['palette'])
                colored_im = cmap(band_scaled)

                # --- Display and save ---
                axes[month-1].imshow(colored_im)
                axes[month-1].set_title(f"{month:02d}")
                axes[month-1].axis('off')

                single_png = os.path.join(out_dir, f"{index}_{stat}_{month:02d}.png")
                plt.imsave(single_png, colored_im)
                print(f"✅ Saved {index} {stat} month {month:02d}")

            except Exception as e:
                print(f"❌ Error month {month:02d}: {e}")
                axes[month-1].axis('off')

        # === Add Horizontal Colorbar at Bottom ===
        norm = mpl.colors.Normalize(vmin=vis_params[index]['min'], vmax=vis_params[index]['max'])
        cmap = ListedColormap(vis_params[index]['palette'])
        sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])

        # Adjust layout to leave space for bottom colorbar
        plt.subplots_adjust(bottom=0.12, top=0.9, wspace=0.05, hspace=0.15)

        # Horizontal colorbar below all images
        cbar_ax = fig.add_axes([0.2, 0.05, 0.6, 0.02])  # [left, bottom, width, height]
        cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
        cbar.set_label(f"{index} Value", fontsize=12)

        plt.suptitle(f"{index} {stat.capitalize()} Composites ({year})", fontsize=16)

        # --- Save summary figure ---
        summary_png = os.path.join(out_dir, f"sentinel2_{index}_{stat}_{year}_summary.png")
        plt.savefig(summary_png, dpi=300)
        plt.close()
        print(f"💾 Saved summary composite: {summary_png}")

    print("\n🎉 All composites and RGB/FalseColor images saved successfully!")


In [ ]:
# For a single Index
print("➡️ About to run composite function...")
save_monthly_composites_with_colorbar(
    collection=s2_sr,
    aoi=aoi_fc.geometry(),
    index='NDVI',
    year=2024,
    scale=10
)


In [ ]:
# Run this for all the Indices
for idx in ["NDVI", "EVI", "VCI"]:
    print(f"\n➡️ Generating composites for {idx}...")
    save_monthly_composites_with_colorbar(
        collection=s2_sr,  # your ImageCollection with indices added
        aoi=aoi_fc.geometry(),
        index=idx,
        year=2024,
        scale=10
    )



➡️ Generating composites for NDVI...
💾 Saving all outputs to: outputs\NDVI_2024

📊 Processing MEAN composites for NDVI...
Generating URL ...
Please wait ...
Data downloaded to C:\Users\PC\AppData\Local\Temp\NDVI_mean_01.tif
✅ Saved NDVI mean month 01
Generating URL ...
Please wait ...
Data downloaded to C:\Users\PC\AppData\Local\Temp\NDVI_mean_02.tif
✅ Saved NDVI mean month 02
Generating URL ...
Please wait ...
Data downloaded to C:\Users\PC\AppData\Local\Temp\NDVI_mean_03.tif
✅ Saved NDVI mean month 03
Generating URL ...
Please wait ...
Data downloaded to C:\Users\PC\AppData\Local\Temp\NDVI_mean_04.tif
✅ Saved NDVI mean month 04
Generating URL ...
Please wait ...
Data downloaded to C:\Users\PC\AppData\Local\Temp\NDVI_mean_05.tif
✅ Saved NDVI mean month 05
Generating URL ...
Please wait ...
Data downloaded to C:\Users\PC\AppData\Local\Temp\NDVI_mean_06.tif
✅ Saved NDVI mean month 06
Generating URL ...
Please wait ...
Data downloaded to C:\Users\PC\AppData\Local\Temp\NDVI_mean_07.tif
✅

In [ ]:
# --- Enhanced GIF creation with titles, month labels, and colorbars ---
import os
from glob import glob
import imageio.v3 as iio
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import ee

def create_monthly_gifs_with_labels(index_list, year, aoi, vis_params_dict):
    """
    Creates enhanced animated GIFs (mean/min/max) for vegetation indices.
    Adds month labels, AOI title, and horizontal colorbar.
    """
    months = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

    # Compute AOI center for labeling
    coords = aoi.centroid().coordinates().getInfo()
    aoi_label = f"AOI Center: ({coords[0]:.2f}, {coords[1]:.2f})"

    for index in index_list:
        out_dir = os.path.join("outputs", f"{index}_{year}")
        if not os.path.exists(out_dir):
            print(f"⚠️ Skipping {index}: folder not found ({out_dir})")
            continue

        print(f"\n🎞️ Creating enhanced GIFs for {index} ({year})...")

        vis = vis_params_dict[index]
        cmap = mpl.colors.ListedColormap(vis["palette"])
        norm = mpl.colors.Normalize(vmin=vis["min"], vmax=vis["max"])

        for stat in ["mean", "min", "max"]:
            pattern = os.path.join(out_dir, f"{index}_{stat}_*.png")
            png_files = sorted(glob(pattern), key=lambda x: int(os.path.basename(x).split("_")[-1].split(".")[0]))

            if len(png_files) != 12:
                print(f"⚠️ Skipping {index}-{stat}: found {len(png_files)} PNGs instead of 12.")
                continue

            frames = []
            for i, png in enumerate(png_files):
                img = plt.imread(png)

                # Create figure with colorbar + annotations
                fig, ax = plt.subplots(figsize=(8, 6))
                ax.imshow(img)
                ax.axis("off")

                # Add labels
                ax.text(0.05, 0.92, f"{months[i]} {year}", transform=ax.transAxes,
                        fontsize=16, fontweight="bold", color="white", bbox=dict(facecolor="black", alpha=0.5, pad=5))
                ax.text(0.05, 0.05, aoi_label, transform=ax.transAxes,
                        fontsize=10, color="white", bbox=dict(facecolor="black", alpha=0.4, pad=3))

                # Add title
                fig.suptitle(f"{index} {stat.capitalize()} Composite ({year})", fontsize=18, fontweight="bold")

                # Add horizontal colorbar below the image
                cax = fig.add_axes([0.2, 0.08, 0.6, 0.03])
                cb = mpl.colorbar.ColorbarBase(cax, cmap=cmap, norm=norm, orientation="horizontal")
                cb.set_label(f"{index} Value", fontsize=12)

                # Convert figure to image array for GIF
                fig.canvas.draw()
                frame = np.array(fig.canvas.renderer.buffer_rgba())
                frames.append(frame)
                plt.close(fig)

            # Save animated GIF
            gif_path = os.path.join(out_dir, f"{index}_{stat}_{year}_enhanced.gif")
            iio.imwrite(gif_path, frames, duration=800, loop=0)
            print(f"✅ Created enhanced GIF: {gif_path}")

    print("\n🎉 All enhanced GIFs created successfully!")


In [ ]:
#  GIF generation for all index
vis_params_dict = {
    "NDVI": {"min": 0, "max": 1, "palette": ["red", "yellow", "green"]},
    "EVI":  {"min": -0.2, "max": 1, "palette": ["blue", "yellow", "green"]},
    "VCI":  {"min": 0, "max": 100, "palette": ["red", "yellow", "green"]}
}

create_monthly_gifs_with_labels(
    index_list=["NDVI", "EVI", "VCI"],
    year=2024,
    aoi=aoi_fc.geometry(),   # your AOI feature or geometry
    vis_params_dict=vis_params_dict
)


In [ ]:
# gif generation for a single Index
create_monthly_gifs_with_labels(
    aoi=aoi_fc.geometry(),
    year=2024,
    indices=["NDVI"],  # 👈 Only one index here
    vis_params_dict = vis_params_dict
    stat="mean"
)


#### Side by Side Multi- GIF Index Maker

In [ ]:
# --- Enhanced Side-by-side GIF creator for NDVI, EVI, VCI ---
import os
from glob import glob
import imageio.v3 as iio
import matplotlib.pyplot as plt
import numpy as np
import ee

def create_side_by_side_index_gif_with_colorbar(aoi, year, indices=["NDVI", "EVI", "VCI"], stat="mean"):
    """
    Creates a side-by-side monthly GIF comparing NDVI, EVI, and VCI.
    Adds colorbars for each index and labels AOI coordinates + year.
    """
    months = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

    vis_params = {
        "NDVI": {"min": 0, "max": 1, "palette": ["red","yellow","green"], "label": "NDVI Value"},
        "EVI":  {"min": -0.2, "max": 1, "palette": ["blue","yellow","green"], "label": "EVI Value"},
        "VCI":  {"min": 0, "max": 100, "palette": ["red","yellow","green"], "label": "VCI (%)"},
    }

    # --- Compute AOI centroid coordinates for display ---
    centroid = aoi.centroid().coordinates().getInfo()
    lon, lat = centroid
    coords_label = f"AOI Center: ({lon:.4f}, {lat:.4f})"

    # --- Check for missing folders ---
    missing = [i for i in indices if not os.path.exists(os.path.join("outputs", f"{i}_{year}"))]
    if missing:
        print(f"⚠️ Missing folders for indices: {missing}")
        return

    out_dir = os.path.join("outputs", f"Comparison_{year}")
    os.makedirs(out_dir, exist_ok=True)
    print(f"\n🎞️ Creating enhanced side-by-side vegetation index GIF ({year}, {stat})...")
    frames = []

    # --- Create frames for each month ---
    for month in range(1, 13):
        imgs = []
        valid = True

        # Load each index image for this month
        for idx in indices:
            path = os.path.join("outputs", f"{idx}_{year}", f"{idx}_{stat}_{month:02d}.png")
            if not os.path.exists(path):
                print(f"⚠️ Missing {path}")
                valid = False
                break
            imgs.append(plt.imread(path))
        if not valid:
            continue

        # --- Plot combined figure ---
        fig, axes = plt.subplots(1, len(indices), figsize=(18, 8))
        if len(indices) == 1:
            axes = [axes]

        for ax, img, idx in zip(axes, imgs, indices):
            ax.imshow(img)
            ax.set_title(f"{idx} – {months[month-1]} {year}", fontsize=14, fontweight="bold")
            ax.axis("off")

            # Colorbar
            norm = plt.Normalize(vmin=vis_params[idx]["min"], vmax=vis_params[idx]["max"])
            cmap = plt.cm.colors.ListedColormap(vis_params[idx]["palette"])
            sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
            cbar = plt.colorbar(
                sm, ax=ax, orientation="horizontal", fraction=0.046, pad=0.08
            )
            cbar.set_label(vis_params[idx]["label"], fontsize=10)
            cbar.ax.tick_params(labelsize=8)

        # --- AOI info below ---
        fig.text(0.5, 0.02, f"{coords_label} | Year: {year}", ha="center", fontsize=11, color="dimgray")

        fig.tight_layout(rect=[0, 0.04, 1, 0.95])
        fig.canvas.draw()
        frame = np.array(fig.canvas.renderer.buffer_rgba())
        frames.append(frame)
        plt.close(fig)

    # --- Save GIF ---
    gif_path = os.path.join(out_dir, f"Enhanced_Veg_Indices_{stat}_{year}.gif")
    iio.imwrite(gif_path, frames, duration=900, loop=0)
    print(f"✅ Enhanced GIF saved: {gif_path}")

    return gif_path


In [ ]:
create_side_by_side_index_gif_with_colorbar(
    aoi=aoi_fc.geometry(),
    year=2024,
    indices=["NDVI", "EVI", "VCI"],
    stat="mean"
)


In [ ]:
# gif generation for a single Index
create_side_by_side_index_gif_with_colorbar(
    aoi=aoi_fc.geometry(),
    year=2024,
    indices=["NDVI"],  # 👈 Only one index here
    vis_params_dict = vis_params_dict
    stat="mean"
)


In [ ]:
print("➡️ About to run composite function...")
save_monthly_composites_with_colorbar(
    collection=s2_sr,
    aoi=aoi_fc.geometry(),
    index='NDVI',
    year=2024,
    scale=10
)
